# 🎨 Rang Manga Editor

अपनी **manga / anime तस्वीर upload करें और लिखकर बताएँ कि क्या बदलना है**: बाल का रंग, चेहरे का भाव, pose, camera angle, रंग भरना, screentone, speech bubble का text हटाना, या दूसरे characters को scene में डालना। बदलाव मनचाहा न हो तो फिर से edit करते रहें।

ये **free GPU** (Kaggle या Google Colab) पर चलता है, फ़ोन के browser से।

**पहली बार:**
1. **GPU चालू करें.** Kaggle (सबसे अच्छा): *Settings → Accelerator → GPU T4 x2* और *Internet → On*.  Colab: *Runtime → Change runtime type → T4 GPU*.
2. **Step 1 ▶** (10–20 मिनट, हर नए session में एक बार)।
3. **Step 2 ▶**: अपनी manga तस्वीर चुनें।
4. **Step 3**: QUICK_EDIT चुनें या CHANGE में लिखें, फिर ▶। *Before / After* नीचे दिखेगा।
5. और बदलना है? CHANGE बदलकर Step 3 ▶ फिर से। हर बार पिछले result पर आगे edit होता है। Undo के लिए Step 4।

| Engine | कहाँ | खास बात |
|---|---|---|
| **Qwen Image Edit 2511** + Lightning | Kaggle (~30 GB RAM) | सबसे अच्छा manga edit, add-ons: Manga tone, Anime, Photo→anime, Camera angle, Noir panel, Sharpen |
| **FLUX 3 Image Klein** | Colab या Kaggle | हल्का, 10 तस्वीरें तक साथ जोड़ सकता है |

`ENGINE = Auto` अपने-आप सही engine चुनता है।

> 🔒 Safety filter हमेशा चालू है: adult/sexual edits और तस्वीरें रोकी जाती हैं।

*Powered by [WanGP](https://github.com/deepbeepmeep/Wan2GP), [Qwen-Image-Edit-2511](https://github.com/QwenLM/Qwen-Image) and [FLUX 3 Image Klein](https://docs.bfl.ml/flux_3/flux3_image_klein). Rang Manga Editor is a free, non-commercial wrapper and is not made or endorsed by those projects. See NOTICE.*


In [ ]:
#@title 1️⃣ Setup — हर नए session में एक बार (10–20 मिनट) / One time per session
#@markdown **USE_GOOGLE_DRIVE** (सिर्फ़ Colab): आपकी edit की हुई तस्वीरें Google Drive के `RangManga` फ़ोल्डर में save रहेंगी।
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}

# Rang Manga Editor — free AI manga picture editor for your own use.
# Powered by WanGP (https://github.com/deepbeepmeep/Wan2GP, WanGP Community License) and
# Qwen-Image-Edit-2511 (Qwen team, Apache-2.0) / FLUX 3 Image Klein (FLUX Kommunity License). Install steps adapted from
# Square-Zero-Labs/Wan2GP-on-Colab (Apache-2.0). See NOTICE in the repository.
import hashlib, json, os, platform, random, re, shutil, subprocess, sys, tarfile, tempfile, time, urllib.request
from pathlib import Path

ON_KAGGLE = Path('/kaggle/working').is_dir()
# Model files are big (Qwen Edit ≈ 35 GB, FLUX 3 Klein ≈ 15 GB), so they go on the large temporary disk.
# Kaggle's /kaggle/working is limited to 20 GB and is used only for finished videos.
WORK = Path('/tmp/rangmanga') if ON_KAGGLE else Path('/content')
WANGP_DIR = WORK / 'WanGP'
CACHE_DIR = WORK / '.rangmanga-cache'
WANGP_REPO = 'https://github.com/deepbeepmeep/Wan2GP.git'
WANGP_COMMIT = 'f0b54e02aa32b539e7be5c65cf4ada67db69da26'  # WanGP v19.13, 6 Nov 2026 (pinned so updates don't break this)

if ON_KAGGLE:
    OUTPUT_DIR = Path('/kaggle/working/RangManga')
elif USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = Path('/content/drive/MyDrive/RangManga')
    except Exception as exc:
        print('⚠️ Google Drive नहीं जुड़ा (' + str(exc)[:120] + '). तस्वीरें सिर्फ़ इस session में रहेंगी।')
        OUTPUT_DIR = Path('/content/RangManga')
else:
    OUTPUT_DIR = Path('/content/RangManga')
SAVED_TO_DRIVE = str(OUTPUT_DIR).startswith('/content/drive/')
for folder in (WORK, OUTPUT_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def run(cmd, **kwargs):
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True, **kwargs)


# ---- 1. GPU ---------------------------------------------------------------
try:
    gpu_info = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        check=True, capture_output=True, text=True).stdout.strip()
except Exception:
    raise SystemExit('❌ GPU नहीं मिला। Colab: Runtime → Change runtime type → T4 GPU → Save. '
                     'Kaggle: Settings → Accelerator → GPU T4 x2, और Internet ON करें।')
print('✅ GPU:', gpu_info)

# ---- 2. WanGP code (fixed version) -----------------------------------------
if not (WANGP_DIR / 'wgp.py').is_file():
    if WANGP_DIR.exists():
        shutil.rmtree(WANGP_DIR)
    WANGP_DIR.mkdir(parents=True)
    run(['git', 'init', '-q', WANGP_DIR])
    run(['git', '-C', WANGP_DIR, 'remote', 'add', 'origin', WANGP_REPO])
    run(['git', '-C', WANGP_DIR, 'fetch', '-q', '--depth', '1', 'origin', WANGP_COMMIT])
    run(['git', '-C', WANGP_DIR, 'checkout', '-q', 'FETCH_HEAD'])
print('✅ WanGP code ready')

# ---- 3. System libraries ---------------------------------------------------
SUDO = ['sudo'] if shutil.which('sudo') and os.geteuid() != 0 else []
APT_PACKAGES = ['libglib2.0-0', 'libgl1', 'libportaudio2']
missing = [p for p in APT_PACKAGES if subprocess.run(['dpkg', '-s', p], capture_output=True).returncode != 0]
if missing:
    apt_env = dict(os.environ, DEBIAN_FRONTEND='noninteractive')
    run(SUDO + ['apt-get', 'update', '-qq'], env=apt_env)
    run(SUDO + ['apt-get', 'install', '-y', '-qq', '--no-install-recommends', *missing], env=apt_env)
print('✅ System libraries ready')

# ---- 4. FFmpeg (needs -fps_mode; older builds are replaced by a checksum-verified static build)
FFMPEG_RELEASE_API = 'https://api.github.com/repos/BtbN/FFmpeg-Builds/releases/latest'
FFMPEG_DOWNLOAD_PREFIX = 'https://github.com/BtbN/FFmpeg-Builds/releases/download/'
FFMPEG_BIN_DIR = WANGP_DIR / 'ffmpeg_bins'


def ffmpeg_ok(binary):
    try:
        out = subprocess.run([str(binary), '-hide_banner', '-h', 'full'], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, timeout=30)
    except (OSError, subprocess.TimeoutExpired):
        return False
    return out.returncode == 0 and re.search(r'^\s*-fps_mode(?:\[:[^]]+\])?(?:\s|$)', out.stdout, re.M) is not None


def works(binary):
    try:
        return subprocess.run([str(binary), '-version'], stdout=subprocess.DEVNULL,
                              stderr=subprocess.DEVNULL, timeout=30).returncode == 0
    except (OSError, subprocess.TimeoutExpired):
        return False


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def install_static_ffmpeg():
    arch = {'x86_64': 'linux64', 'amd64': 'linux64', 'aarch64': 'linuxarm64', 'arm64': 'linuxarm64'}.get(platform.machine().lower())
    if not arch:
        raise SystemExit('❌ इस मशीन के लिए FFmpeg build नहीं मिला।')
    req = urllib.request.Request(FFMPEG_RELEASE_API, headers={'Accept': 'application/vnd.github+json', 'User-Agent': 'RangManga'})
    with urllib.request.urlopen(req, timeout=30) as resp:
        release = json.load(resp)
    pattern = re.compile(rf'^ffmpeg-n(?P<v>\d+\.\d+)-latest-{arch}-gpl-(?P=v)\.tar\.xz$')
    candidates = []
    for asset in release.get('assets', []):
        m = pattern.fullmatch(asset.get('name', ''))
        if m:
            candidates.append((tuple(int(x) for x in m.group('v').split('.')), asset))
    if not candidates:
        raise SystemExit('❌ FFmpeg का stable build नहीं मिला।')
    asset = max(candidates, key=lambda c: c[0])[1]
    url, digest = asset.get('browser_download_url', ''), asset.get('digest') or ''
    if not url.startswith(FFMPEG_DOWNLOAD_PREFIX) or not digest.startswith('sha256:'):
        raise SystemExit('❌ FFmpeg download की जाँच नहीं हो सकी, इसलिए रोका गया।')
    archive = CACHE_DIR / asset['name']
    if not (archive.is_file() and sha256_of(archive) == digest.split(':', 1)[1].lower()):
        print('Downloading FFmpeg:', asset['name'], flush=True)
        tmp = archive.with_suffix('.download')
        req = urllib.request.Request(url, headers={'User-Agent': 'RangManga'})
        with urllib.request.urlopen(req, timeout=300) as resp, open(tmp, 'wb') as out:
            shutil.copyfileobj(resp, out)
        if sha256_of(tmp) != digest.split(':', 1)[1].lower():
            tmp.unlink(missing_ok=True)
            raise SystemExit('❌ FFmpeg फ़ाइल का checksum मेल नहीं खाया, इसलिए रोका गया।')
        os.replace(tmp, archive)
    FFMPEG_BIN_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, mode='r:xz') as tar:
        found = {}
        for member in tar.getmembers():
            p = Path(member.name)
            if member.isfile() and p.parent.name == 'bin' and p.name in ('ffmpeg', 'ffprobe'):
                found[p.name] = member
        if set(found) != {'ffmpeg', 'ffprobe'}:
            raise SystemExit('❌ FFmpeg archive अधूरा है।')
        for name, member in found.items():
            src = tar.extractfile(member)
            dest = FFMPEG_BIN_DIR / name
            with src, open(dest, 'wb') as out:
                shutil.copyfileobj(src, out)
            dest.chmod(0o755)
    return FFMPEG_BIN_DIR / 'ffmpeg', FFMPEG_BIN_DIR / 'ffprobe'


local_ff, local_fp = FFMPEG_BIN_DIR / 'ffmpeg', FFMPEG_BIN_DIR / 'ffprobe'
sys_ff, sys_fp = shutil.which('ffmpeg'), shutil.which('ffprobe')
if local_ff.exists() and ffmpeg_ok(local_ff) and works(local_fp):
    ffmpeg_bin, ffprobe_bin = local_ff, local_fp
elif sys_ff and sys_fp and ffmpeg_ok(sys_ff) and works(sys_fp):
    ffmpeg_bin, ffprobe_bin = Path(sys_ff), Path(sys_fp)
else:
    ffmpeg_bin, ffprobe_bin = install_static_ffmpeg()
if not (ffmpeg_ok(ffmpeg_bin) and works(ffprobe_bin)):
    raise SystemExit('❌ FFmpeg ठीक से काम नहीं कर रहा।')
os.environ['FFMPEG_BINARY'] = str(Path(ffmpeg_bin).resolve())
os.environ['FFPROBE_BINARY'] = str(Path(ffprobe_bin).resolve())
bin_dir = str(Path(ffmpeg_bin).resolve().parent)
os.environ['PATH'] = os.pathsep.join([bin_dir] + [p for p in os.environ.get('PATH', '').split(os.pathsep) if p != bin_dir])
print('✅ FFmpeg ready')

# ---- 5. Python packages (keeps the runtime's PyTorch when it already works) --
PYTORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu128'
REPLACEMENT_TORCH = ('torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0')
UNSUPPORTED_TORCH = ('2.8.0', '2.9.')
TORCH_PROBE = '''
import json
info = {}
try:
    import importlib.metadata as md, torch
    info["torch"] = md.version("torch")
    info["cuda"] = torch.cuda.is_available()
    info["device"] = torch.cuda.get_device_name(0) if info["cuda"] else ""
except Exception:
    pass
print(json.dumps(info))
'''


def probe(code):
    res = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)
    try:
        return json.loads(res.stdout.strip().splitlines()[-1])
    except Exception:
        return {}


def uv_install(*args):
    run([sys.executable, '-m', 'uv', 'pip', 'install', '--system', *args])


run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'])
torch_version = probe(TORCH_PROBE).get('torch', '')
try:
    torch_release = tuple(int(x) for x in torch_version.split('.')[:2])
except ValueError:
    torch_release = ()
if torch_release >= (2, 7) and not torch_version.startswith(UNSUPPORTED_TORCH):
    print('Using the installed PyTorch', torch_version)
else:
    print('Installing PyTorch…', flush=True)
    uv_install(*REPLACEMENT_TORCH, '--index-url', PYTORCH_INDEX_URL)
print('Installing WanGP packages (कुछ मिनट लगेंगे)…', flush=True)
uv_install('-r', WANGP_DIR / 'requirements.txt', '--index-strategy', 'unsafe-best-match')
try:
    uv_install('onnxruntime-gpu==1.22.0')
except subprocess.CalledProcessError:
    uv_install('onnxruntime>=1.22.0')

report = probe(TORCH_PROBE)
if not report.get('torch'):
    raise SystemExit('❌ PyTorch नहीं चल रहा। Runtime → Restart session करें, फिर Step 1 दोबारा चलाएँ।')
if not report.get('cuda'):
    raise SystemExit('❌ GPU नहीं दिख रहा। Runtime → Change runtime type → T4 GPU चुनें, फिर Step 1 दोबारा।')

# ---- 6. Memory profile: Kaggle has ~29 GB RAM (faster profile), free Colab ~12 GB --
RAM_GB = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024 ** 3
WANGP_PROFILE = '4' if RAM_GB >= 24 else '5'


def free_disk_gb(path=WORK):
    return shutil.disk_usage(path).free / 1024 ** 3


# ---- 7. Editor settings and helpers used by the next steps -------------------
HF = 'https://huggingface.co/'
LIGHTNING = {
    4: HF + 'DeepBeepMeep/Qwen_image/resolve/main/loras_accelerators/Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors',
    8: HF + 'DeepBeepMeep/Qwen_image/resolve/main/loras_accelerators/Qwen-Image-Edit-2511-Lightning-8steps-V1.0-bf16.safetensors',
}
# Manga / anime add-ons (LoRAs) for Qwen Edit 2511. List taken from
# PRITHIVSAKTHIUR/Qwen-Image-Edit-2511-LoRAs-Fast-Lazy-Load (Apache-2.0). Each LoRA keeps its own license.
STYLE_LORAS = {
    'Manga tone (screentone)': HF + 'nappa114514/Qwen-Image-Edit-2509-Manga-Tone/resolve/main/tone001.safetensors',
    'Anime':                   HF + 'prithivMLmods/Qwen-Image-Edit-2511-Anime/resolve/main/Qwen-Image-Edit-2511-Anime-2000.safetensors',
    'Photo to anime':          HF + 'autoweeb/Qwen-Image-Edit-2509-Photo-to-Anime/resolve/main/Qwen-Image-Edit-2509-Photo-to-Anime_000001000.safetensors',
    'Camera angle':            HF + 'fal/Qwen-Image-Edit-2511-Multiple-Angles-LoRA/resolve/main/qwen-image-edit-2511-multiple-angles-lora.safetensors',
    'Noir comic panel':        HF + 'prithivMLmods/Qwen-Image-Edit-2511-Noir-Comic-Book-Panel/resolve/main/Noir-Comic-Book-Panel_20.safetensors',
    'Unblur / sharpen':        HF + 'prithivMLmods/Qwen-Image-Edit-2511-Unblur-Upscale/resolve/main/Qwen-Image-Edit-Unblur-Upscale_15.safetensors',
}
ENGINES = {
    'qwen': {'label': 'Qwen Image Edit 2511 (manga add-ons)', 'model_type': 'qwen_image_edit_plus2_20B',
             'min_ram': 24, 'min_disk': 45, 'max_refs': 3, 'eta': 'अनुमान 2–5 मिनट हर तस्वीर (free T4)'},
    'klein': {'label': 'FLUX 3 Image Klein', 'model_type': 'flux3_klein',
              'min_ram': 0, 'min_disk': 25, 'max_refs': 10, 'eta': 'अनुमान 1–3 मिनट हर तस्वीर (free T4)'},
}
SIZES = {'Normal (~1 MP)': 1024, 'Fast (~0.6 MP)': 768}
UPLOADS_DIR = OUTPUT_DIR / 'uploads'
EDITS_DIR = OUTPUT_DIR / 'edits'
HISTORY_FILE = OUTPUT_DIR / 'history.json'
IMAGE_EXTS = ('.png', '.jpg', '.jpeg', '.webp')
for folder in (UPLOADS_DIR, EDITS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def save_upload(raw, dest):
    """Checks the bytes are a picture and stores it as PNG (max 2048 px)."""
    import io
    from PIL import Image
    if len(raw) > 25 * 1024 * 1024:
        raise ValueError('फ़ोटो बहुत बड़ी है (25 MB से कम रखें)।')
    try:
        with Image.open(io.BytesIO(raw)) as im:
            im = im.convert('RGB')
            im.thumbnail((2048, 2048))
            im.save(dest)
    except ValueError:
        raise
    except Exception:
        raise ValueError('ये फ़ाइल फ़ोटो नहीं लगती। JPG, PNG या WEBP डालें।')
    return Path(dest)


def fetch_url(url):
    if not url.startswith('https://'):
        raise ValueError('फ़ोटो का पूरा https:// लिंक डालें।')
    req = urllib.request.Request(url, headers={'User-Agent': 'RangManga'})
    with urllib.request.urlopen(req, timeout=60) as resp:
        return resp.read(25 * 1024 * 1024 + 1)


def load_history():
    try:
        return json.loads(HISTORY_FILE.read_text(encoding='utf-8'))
    except Exception:
        return []


def save_history(items):
    tmp = HISTORY_FILE.with_suffix('.tmp')
    tmp.write_text(json.dumps(items, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(tmp, HISTORY_FILE)


# ---- safety filter: adult/sexual content is blocked in prompts, inputs and results ----
BLOCKED_WORDS = re.compile(
    r'\b(nsfw|nude|nudity|naked|topless|bottomless|undress\w*|stripping|striptease|lingerie|underwear|bikini|sex\w*|porn\w*|'
    r'hentai|ecchi|erotic\w*|explicit|genital\w*|nipple\w*|breasts?|boobs?|cleavage|seductive|lewd|fetish\w*|'
    r'नंगा|नंगी|नग्न|सेक्स|अश्लील)\b', re.IGNORECASE)
_safety_model = None


def prompt_is_blocked(text):
    return bool(BLOCKED_WORDS.search(text or ''))


def image_is_adult(path):
    """True when the picture looks sexual/adult. Fails closed (blocks) if the check cannot run."""
    global _safety_model
    try:
        if _safety_model is None:
            from transformers import pipeline
            _safety_model = pipeline('image-classification', model='Falconsai/nsfw_image_detection', device=-1)
        from PIL import Image
        with Image.open(path) as im:
            scores = {r['label'].lower(): r['score'] for r in _safety_model(im.convert('RGB'))}
        return scores.get('nsfw', 0.0) >= 0.5
    except Exception as exc:
        print('⚠️ Safety check नहीं चल पाया (' + str(exc)[:100] + '), इसलिए तस्वीर रोकी गई।')
        return True


def run_wangp(tasks, out_dir):
    """Runs WanGP without a web UI on a list of settings and returns the new pictures in order."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    job_file = out_dir / 'tasks.json'
    job_file.write_text(json.dumps(tasks, ensure_ascii=False, indent=2), encoding='utf-8')
    env = os.environ.copy()
    env.update({'HF_HOME': str(CACHE_DIR / 'huggingface'), 'TORCH_HOME': str(CACHE_DIR / 'torch'),
                'PYTHONUNBUFFERED': '1'})
    cmd = [sys.executable, '-u', 'wgp.py', '--process', str(job_file), '--output-dir', str(out_dir),
           '--profile', WANGP_PROFILE]
    proc = subprocess.Popen(cmd, cwd=str(WANGP_DIR), env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    last_line, last_bar = '', 0.0
    for line in proc.stdout:
        line = line.rstrip()
        if not line or line == last_line:
            continue
        is_bar = '%|' in line or 'it/s' in line or 's/it' in line
        if is_bar and time.time() - last_bar < 10:
            continue  # keep progress bars from flooding the page
        if is_bar:
            last_bar = time.time()
        print(line, flush=True)
        last_line = line
    proc.wait()
    pictures = sorted((p for p in out_dir.rglob('*') if p.suffix.lower() in IMAGE_EXTS),
                      key=lambda p: p.stat().st_mtime)
    return proc.returncode, pictures


def thumb_html(path, title, width=320):
    import base64, io
    from PIL import Image
    with Image.open(path) as im:
        im = im.convert('RGB')
        im.thumbnail((900, 900))
        buf = io.BytesIO()
        im.save(buf, 'JPEG', quality=85)
    data = base64.b64encode(buf.getvalue()).decode()
    return (f'<figure style="margin:4px;display:inline-block;vertical-align:top;max-width:{width}px">'
            f'<img src="data:image/jpeg;base64,{data}" style="width:100%;border-radius:8px">'
            f'<figcaption style="font:13px sans-serif">{title}</figcaption></figure>')


STATE_FILE = OUTPUT_DIR / 'state.json'


def load_state():
    try:
        return json.loads(STATE_FILE.read_text(encoding='utf-8'))
    except Exception:
        return {'main': None, 'extras': [], 'last': None}


def save_state(state):
    tmp = STATE_FILE.with_suffix('.tmp')
    tmp.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(tmp, STATE_FILE)


SETUP_DONE = True
print(f"\n✅ Setup पूरा! PyTorch {report['torch']} on {report['device']} · RAM {RAM_GB:.0f} GB · "
      f"disk free {free_disk_gb():.0f} GB · profile {WANGP_PROFILE}")
print('तस्वीरें यहाँ save होंगी:', OUTPUT_DIR)
if RAM_GB < ENGINES['qwen']['min_ram']:
    print('ℹ️ यहाँ "FLUX 3 Klein" चलेगा। manga add-ons वाले Qwen Edit के लिए Kaggle इस्तेमाल करें।')
print('अब Step 2 में अपनी manga तस्वीर upload करें।')


In [ ]:
#@title 2️⃣ तस्वीर upload करें / Upload a picture
#@markdown - **Main picture**: वो manga तस्वीर जिसे edit करना है।
#@markdown - **Extra pictures** (optional): characters या चीज़ें जिन्हें main picture में डालना है (Qwen: 2 तक, Klein: 9 तक)।
#@markdown
#@markdown ▶ चलाएँ और फ़ोटो चुनें। या **PICTURE_URL** में फ़ोटो का https:// लिंक डालें।
WHAT = "Main picture (जिसे edit करना है)"  #@param ["Main picture (जिसे edit करना है)", "Extra pictures (characters / objects)"]
PICTURE_URL = ""  #@param {type:"string"}

import time
from pathlib import Path
from IPython.display import HTML, display

if not globals().get('SETUP_DONE'):
    raise SystemExit('पहले Step 1 (Setup) चलाएँ।')
kind = 'main' if WHAT.startswith('Main') else 'extra'


def store_pictures(items):
    """items: list of (name, raw bytes). Saves, checks safety, updates the saved state."""
    state = load_state()
    saved = []
    for name, raw in items:
        dest = UPLOADS_DIR / f"{kind}_{time.strftime('%Y%m%d_%H%M%S')}_{len(saved)}.png"
        try:
            save_upload(raw, dest)
        except ValueError as exc:
            print(f'❌ {name}: {exc}')
            continue
        if image_is_adult(dest):
            dest.unlink(missing_ok=True)
            print(f'❌ {name}: ये तस्वीर safety filter ने रोकी (adult content)।')
            continue
        saved.append(str(dest))
    if not saved:
        return state
    if kind == 'main':
        state['main'] = saved[0]
        state['extras'] = []   # a new main picture starts a fresh set of extras
        state['last'] = None   # and a fresh edit chain
        print('✅ Main picture तैयार। अब Step 3 में लिखें कि क्या बदलना है।')
    else:
        state['extras'] = (state.get('extras') or []) + saved
        state['extras'] = state['extras'][-9:]
        print(f"✅ {len(saved)} extra picture(s) जुड़ीं। कुल extra: {len(state['extras'])}")
    save_state(state)
    display(HTML(''.join(thumb_html(p, Path(p).name, 200) for p in saved)))
    return state


if PICTURE_URL.strip():
    try:
        store_pictures([(PICTURE_URL.strip(), fetch_url(PICTURE_URL.strip()))])
    except Exception as exc:
        raise SystemExit(f'❌ लिंक से फ़ोटो नहीं मिली: {exc}')
else:
    try:
        from google.colab import files as colab_files
    except ImportError:
        colab_files = None
    if colab_files is not None:
        picked = colab_files.upload()
        if not picked:
            raise SystemExit('कोई फ़ोटो नहीं चुनी गई।')
        items = list(picked.items())
        store_pictures(items[:1] if kind == 'main' else items[:9])
    else:
        # Kaggle / Jupyter: upload button. The picture is saved as soon as it is chosen.
        import ipywidgets as widgets
        button = widgets.FileUpload(accept='image/*', multiple=(kind == 'extra'),
                                    description='Choose picture' if kind == 'main' else 'Choose pictures')
        log = widgets.Output()

        def on_upload(change):
            value = change['new']
            if isinstance(value, dict):          # ipywidgets 7: {name: {'content': bytes, ...}}
                items = [(n, bytes(v['content'])) for n, v in value.items()]
            else:                                # ipywidgets 8: ({'name':..., 'content': memoryview}, ...)
                items = [(v['name'], bytes(v['content'])) for v in value]
            if not items:
                return
            with log:
                store_pictures(items[:1] if kind == 'main' else items[:9])

        button.observe(on_upload, names='value')
        display(button, log)
        print('ऊपर बटन दबाकर फ़ोटो चुनें। "✅" दिखने के बाद Step 3 चलाएँ।')


In [ ]:
#@title 3️⃣ Edit करें / Edit the picture
#@markdown ### किस तस्वीर पर
START_FROM = "Last result (आगे edit करते रहें)"  #@param ["Last result (आगे edit करते रहें)", "Uploaded main picture"]
#@markdown ### क्या करना है
#@markdown **QUICK_EDIT** चुनें, या "Custom" चुनकर **CHANGE** में अपनी बात लिखें (English में सबसे अच्छा)।
QUICK_EDIT = "Custom (CHANGE में लिखें)"  #@param ["Custom (CHANGE में लिखें)", "Colorize manga (रंग भरें)", "Black & white manga + screentone", "Remove text from speech bubbles", "Photo → anime", "Anime style", "Change expression (CHANGE में भाव लिखें)", "Change pose / action (CHANGE में लिखें)", "Change camera angle (CHANGE में लिखें)", "Put extra pictures into the scene", "Noir comic panel", "Sharpen / unblur"]
CHANGE = "Make her hair red and add a small smile"  #@param {type:"string"}
#@markdown **KEEP_STYLE**: बाकी सब (lines, चेहरे, background) वैसा ही रखने को कहें।
KEEP_STYLE = True  #@param {type:"boolean"}
#@markdown **USE_EXTRA_PICTURES**: Custom edit में भी extra pictures भेजें ("Put extra pictures" में ये हमेशा भेजी जाती हैं)।
USE_EXTRA_PICTURES = False  #@param {type:"boolean"}
#@markdown ### Engine
ENGINE = "Auto"  #@param ["Auto", "Qwen Edit 2511 · manga add-ons (Kaggle)", "FLUX 3 Klein (Colab या Kaggle)"]
SPEED = "Balanced (8 steps)"  #@param ["Balanced (8 steps)", "Fast (4 steps)"]
SIZE = "Normal (~1 MP)"  #@param ["Normal (~1 MP)", "Fast (~0.6 MP)"]
VERSIONS = 1  #@param {type:"slider", min:1, max:2, step:1}
SEED = -1  #@param {type:"integer"}

import random, shutil, time
from pathlib import Path
from IPython.display import HTML, display

if not globals().get('SETUP_DONE'):
    raise SystemExit('पहले Step 1 (Setup) चलाएँ।')

# ---- engine -----------------------------------------------------------------
if ENGINE.startswith('Qwen'):
    engine = 'qwen'
elif ENGINE.startswith('FLUX'):
    engine = 'klein'
else:
    engine = 'qwen' if RAM_GB >= ENGINES['qwen']['min_ram'] else 'klein'
eng = ENGINES[engine]
if RAM_GB < eng['min_ram']:
    raise SystemExit(f'Qwen Edit को ~30 GB RAM चाहिए, यहाँ {RAM_GB:.0f} GB है। Kaggle पर चलाएँ, '
                     'या ENGINE में "FLUX 3 Klein" चुनें।')
if free_disk_gb() < eng['min_disk']:
    raise SystemExit(f'Disk में जगह कम है ({free_disk_gb():.0f} GB खाली, {eng["min_disk"]} GB चाहिए)।')

# ---- which picture -----------------------------------------------------------
state = load_state()
source = None
if START_FROM.startswith('Last') and state.get('last') and Path(state['last']).is_file():
    source = state['last']
elif state.get('main') and Path(state['main']).is_file():
    if START_FROM.startswith('Last'):
        print('ℹ️ अभी कोई edit नहीं हुआ, इसलिए upload की हुई main picture से शुरू कर रहे हैं।')
    source = state['main']
if not source:
    raise SystemExit('पहले Step 2 में main picture upload करें।')

# ---- what to do --------------------------------------------------------------
change = CHANGE.strip().rstrip('. ')
QUICK = {
    'Custom':              ('{change}', None, True),
    'Colorize manga':      ('Colorize this manga image with clean, vibrant anime colors. {change}', None, False),
    'Black & white manga': ('Paint with manga tone.', 'Manga tone (screentone)', False),
    'Remove text':         ('Remove all text from the speech bubbles and captions, leaving the bubbles clean and empty.', None, False),
    'Photo → anime':       ('Transform into anime. {change}', 'Photo to anime', False),
    'Anime style':         ('Transform into anime. {change}', 'Anime', False),
    'Change expression':   ("Change the character's facial expression to {change}. Keep the same character, outfit and pose.", None, True),
    'Change pose':         ('Make the character {change}. Keep the same character design, outfit and background.', None, True),
    'Change camera angle': ('{change}', 'Camera angle', False),
    'Put extra pictures':  ('Place the characters or objects from the other pictures into the first picture: {change}. '
                            "Keep the first picture's background, lighting and art style, and draw everyone in the same manga style.", None, False),
    'Noir comic panel':    ('Transform into a noir comic book panel. {change}', 'Noir comic panel', False),
    'Sharpen':             ('Unblur and sharpen the image, keeping every detail of the drawing. {change}', 'Unblur / sharpen', False),
}
key = next(k for k in QUICK if QUICK_EDIT.startswith(k))
template, lora, needs_change = QUICK[key]
if needs_change and not change:
    raise SystemExit('CHANGE में लिखें कि क्या बदलना है (जैसे "Make her hair red")।')
extras = []
if key == 'Put extra pictures' or (key == 'Custom' and USE_EXTRA_PICTURES):
    extras = [p for p in (state.get('extras') or []) if Path(p).is_file()]
    if key == 'Put extra pictures' and not extras:
        raise SystemExit('पहले Step 2 में "Extra pictures" upload करें (जिन characters को डालना है)।')
    if len(extras) > eng['max_refs'] - 1:
        print(f'ℹ️ {eng["label"]} एक साथ {eng["max_refs"] - 1} extra picture तक लेता है; बाकी छोड़ दीं।')
        extras = extras[:eng['max_refs'] - 1]
if key == 'Change camera angle' and not change:
    change = 'Rotate the camera 45 degrees to the left'
prompt = ' '.join(template.format(change=change).split())
if prompt and prompt[-1] not in '.!?':
    prompt += '.'
if KEEP_STYLE and key not in ('Photo → anime', 'Anime style', 'Noir comic panel'):
    prompt += (' Keep the original manga art style, line art, characters, faces and composition'
               ' unchanged except for the requested change.')
if prompt_is_blocked(prompt):
    raise SystemExit('❌ ये edit safety filter ने रोका (adult content)। दूसरा बदलाव लिखें।')
if lora and engine != 'qwen':
    print(f'ℹ️ "{lora}" add-on सिर्फ़ Qwen Edit में है; FLUX 3 Klein सिर्फ़ prompt से करेगा।')
    lora = None

# ---- settings for WanGP ------------------------------------------------------
size = SIZES.get(SIZE, 1024)
versions = max(1, min(2, int(VERSIONS)))
seed = int(SEED) if SEED >= 0 else random.randint(0, 2_000_000_000)
task = {
    'model_type': eng['model_type'],
    'prompt': prompt,
    'image_mode': 1,
    'image_refs': [str(source)] + [str(p) for p in extras],
    'video_prompt_type': 'KI',            # first picture = the one being edited
    'remove_background_images_ref': 0,
    'resolution': f'{size}x{size}',      # pixel budget; the shape follows the first picture
    'batch_size': versions,
    'seed': seed,
}
if engine == 'qwen':
    steps = 4 if SPEED.startswith('Fast') else 8
    loras = [LIGHTNING[steps]] + ([STYLE_LORAS[lora]] if lora else [])
    task.update({'negative_prompt': '', 'num_inference_steps': steps, 'guidance_scale': 1,
                 'activated_loras': loras, 'loras_multipliers': ' '.join('1' for _ in loras)})
else:
    task.update({'num_inference_steps': 16, 'guidance_scale': 1})

job_dir = CACHE_DIR / 'jobs' / time.strftime('%Y%m%d-%H%M%S')
print(f'🎨 {eng["label"]} · {QUICK_EDIT.split(" (")[0]}' + (f' · add-on: {lora}' if lora else ''))
print(f'   Prompt: {prompt}')
print(f'   {eng["eta"]}। पहली बार model download (कई GB) में 10–20 मिनट और लगेंगे। इस tab को खुला रखें।\n', flush=True)
started = time.time()
code, pictures = run_wangp([task], job_dir)
minutes = (time.time() - started) / 60
if code != 0 or not pictures:
    raise SystemExit(f'❌ तस्वीर नहीं बनी (code {code}, {minutes:.1f} मिनट)। ऊपर का error देखें। '
                     'अक्सर हल: SIZE = "Fast", VERSIONS = 1, या Runtime → Restart session करके Step 1 फिर चलाएँ।')

# ---- check, save, show -------------------------------------------------------
history = load_history()
saved, blocked = [], 0
for i, pic in enumerate(pictures[-versions:]):
    if image_is_adult(pic):
        blocked += 1
        continue
    number = len(history) + 1
    dest = EDITS_DIR / f'edit_{number:03d}.png'
    from PIL import Image
    with Image.open(pic) as im:
        im.convert('RGB').save(dest)
    item = {'n': number, 'file': dest.name, 'from': Path(source).name, 'prompt': prompt, 'quick': key,
            'engine': engine, 'lora': lora, 'seed': seed + i, 'extras': len(extras), 'created': time.time()}
    history.append(item)
    saved.append(item)
if blocked:
    print(f'⚠️ {blocked} तस्वीर safety filter ने रोकी (adult content)।')
if not saved:
    raise SystemExit('कोई तस्वीर नहीं बची। दूसरा बदलाव लिखकर फिर कोशिश करें।')
save_history(history)
state['last'] = str(EDITS_DIR / saved[0]['file'])
save_state(state)

print(f'\n✅ तैयार! {minutes:.1f} मिनट')
cards = [thumb_html(source, 'पहले / Before')]
cards += [thumb_html(EDITS_DIR / s['file'], f"#{s['n']} बाद / After · seed {s['seed']}") for s in saved]
display(HTML(''.join(cards)))
if SAVED_TO_DRIVE:
    print('📁 Google Drive → RangManga → edits')
else:
    try:
        from google.colab import files
        for s in saved:
            files.download(str(EDITS_DIR / s['file']))
    except ImportError:
        print('Kaggle: दाईं ओर Output → RangManga → edits से डाउनलोड करें।')
if len(saved) > 1:
    print('ℹ️ "Last result" पहला version (#%d) है। दूसरा चाहिए तो Step 4 में उसका नंबर चुनें।' % saved[0]['n'])
print('\nऔर बदलना है? CHANGE में अगली बात लिखें और ▶ फिर दबाएँ — ये पिछले result पर ही आगे edit करेगा।')


## 💡 Tips

- **English में लिखें**, छोटा और साफ़: *"Make her hair red"*, *"Give him a sword in his right hand"*, *"Change the background to a night city"*.
- **एक बार में एक बदलाव** करें। फिर अगला। हर edit पिछले result पर होता है।
- **Characters को scene में डालना:** Step 2 में पहले कमरे/scene की main picture, फिर WHAT = *Extra pictures* में characters upload करें। Step 3 में QUICK_EDIT = *Put extra pictures into the scene* और CHANGE में लिखें कि वो क्या कर रहे हैं (जैसे *"both sitting on the sofa, talking"*)।
- **बात न माने तो:** VERSIONS = 2 करें, या SEED बदलें, या CHANGE को और साफ़ लिखें।
- **Error / धीमा हो तो:** SIZE = *Fast*, SPEED = *Fast (4 steps)*।
- **Free limits:** Kaggle पर हफ़्ते में ~30 घंटे GPU (phone number verify करना पड़ता है)। Colab free में रोज़ कुछ घंटे।


In [ ]:
#@title 4️⃣ मेरी edits / History, undo और ZIP
#@markdown **USE_EDIT_NUMBER**: किसी पुरानी edit (#नंबर) से आगे edit करना हो (undo) तो उसका नंबर डालें। 0 = कुछ नहीं।
USE_EDIT_NUMBER = 0  #@param {type:"integer"}
SHOW_LAST = 12  #@param {type:"slider", min:4, max:40, step:4}
DOWNLOAD_ZIP = False  #@param {type:"boolean"}

import shutil
from pathlib import Path
from IPython.display import HTML, display

if not globals().get('SETUP_DONE'):
    raise SystemExit('पहले Step 1 चलाएँ।')
history = load_history()
if not history:
    raise SystemExit('अभी कोई edit नहीं हुई।')

if USE_EDIT_NUMBER:
    match = next((h for h in history if h['n'] == int(USE_EDIT_NUMBER)), None)
    if not match or not (EDITS_DIR / match['file']).is_file():
        raise SystemExit(f'#{USE_EDIT_NUMBER} नहीं मिली।')
    state = load_state()
    state['last'] = str(EDITS_DIR / match['file'])
    save_state(state)
    print(f'✅ अब Step 3 ("Last result") #{match["n"]} से आगे edit करेगा।')

items = [h for h in history if (EDITS_DIR / h['file']).is_file()][-int(SHOW_LAST):]
display(HTML(''.join(thumb_html(EDITS_DIR / h['file'], f"#{h['n']} · {h['quick']} · {h['prompt'][:60]}", 220)
                     for h in reversed(items))))

if DOWNLOAD_ZIP:
    zip_path = Path(shutil.make_archive(str(WORK / 'RangManga_edits'), 'zip', str(EDITS_DIR)))
    print(f'{len(history)} edits → {zip_path.name}')
    try:
        from google.colab import files
        files.download(str(zip_path))
    except ImportError:
        out = Path('/kaggle/working') / zip_path.name
        shutil.copy2(zip_path, out)
        print('Kaggle: दाईं ओर Output से', out.name, 'डाउनलोड करें।')
